In [4]:
import warnings
warnings.filterwarnings("ignore")
import joblib

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier


print("=" * 70)
print("WEEKEND ACTIVITY PLANNER - AI RECOMMENDER")
print("=" * 70)


# ============================================================
# 1. LOAD DATA
# ============================================================

users = pd.read_csv("users.csv")
activities = pd.read_csv("activities.csv")
interactions = pd.read_csv("interactions.csv")
weather = pd.read_csv("weather_suitability.csv")


print("\nDataset Shapes")
print("Users:", users.shape)
print("Activities:", activities.shape)
print("Interactions:", interactions.shape)
print("Weather:", weather.shape)


# ============================================================
# 2. CLEAN COLUMN NAMES
# ============================================================

def clean_columns(df):
    df = df.copy()

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )

    return df


users = clean_columns(users)
activities = clean_columns(activities)
interactions = clean_columns(interactions)
weather = clean_columns(weather)


# ============================================================
# 3. BASIC CLEANING
# ============================================================

for df in [users, activities, interactions, weather]:

    for col in df.columns:

        if df[col].dtype == "object":
            df[col] = df[col].fillna("").astype(str).str.strip()


# numeric columns

numeric_activity_cols = [
    "estimated_cost_usd_min",
    "estimated_cost_usd_max",
    "typical_duration_min",
    "lat",
    "lon",
    "data_quality_score"
]

for col in numeric_activity_cols:

    if col in activities.columns:
        activities[col] = pd.to_numeric(
            activities[col],
            errors="coerce"
        )


activities["estimated_cost_usd_min"] = activities[
    "estimated_cost_usd_min"
].fillna(0)

activities["estimated_cost_usd_max"] = activities[
    "estimated_cost_usd_max"
].fillna(
    activities["estimated_cost_usd_min"]
)

activities["avg_cost"] = (
    activities["estimated_cost_usd_min"] +
    activities["estimated_cost_usd_max"]
) / 2


# ============================================================
# 4. BOOLEAN NORMALIZATION
# ============================================================

def to_bool(x):

    if pd.isna(x):
        return False

    if isinstance(x, (bool, np.bool_)):
        return bool(x)

    return str(x).strip().lower() in [
        "true",
        "true",
        "yes",
        "y",
        "1"
    ]


if "has_kids" in users.columns:
    users["has_kids_bool"] = users["has_kids"].apply(to_bool)
else:
    users["has_kids_bool"] = False


if "wheelchair_need" in users.columns:
    users["wheelchair_need_bool"] = users[
        "wheelchair_need"
    ].apply(to_bool)
else:
    users["wheelchair_need_bool"] = False


if "wheelchair" in activities.columns:
    activities["wheelchair_bool"] = activities[
        "wheelchair"
    ].apply(to_bool)
else:
    activities["wheelchair_bool"] = False


if "family_kids_suitable" in activities.columns:
    activities["kids_suitable_bool"] = activities[
        "family_kids_suitable"
    ].apply(to_bool)
else:
    activities["kids_suitable_bool"] = False


# ============================================================
# 5. NORMALIZE TEXT FEATURES
# ============================================================

for col in [
    "category",
    "subcategory",
    "city",
    "state",
    "indoor_outdoor",
    "group_suitability"
]:

    if col in activities.columns:

        activities[col] = (
            activities[col]
            .fillna("")
            .astype(str)
            .str.lower()
            .str.strip()
        )


for col in [
    "home_city",
    "home_state",
    "group_type",
    "indoor_outdoor_preference"
]:

    if col in users.columns:

        users[col] = (
            users[col]
            .fillna("")
            .astype(str)
            .str.lower()
            .str.strip()
        )


# ============================================================
# 6. INTERACTION CLEANING
# ============================================================

interactions["event_date"] = pd.to_datetime(
    interactions["event_date"],
    errors="coerce"
)

interactions["rating"] = pd.to_numeric(
    interactions["rating"],
    errors="coerce"
)

interactions["rating"] = interactions[
    "rating"
].fillna(0)


interactions["interaction_type"] = (
    interactions["interaction_type"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)


interactions["weather_context"] = (
    interactions["weather_context"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)


interactions["group_type"] = (
    interactions["group_type"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.strip()
)


# ============================================================
# 7. MERGE INTERACTIONS + USERS + ACTIVITIES
# ============================================================

print("\nPreparing recommendation dataset...")


data = interactions.merge(
    users,
    on="user_id",
    how="left",
    suffixes=("", "_user")
)


data = data.merge(
    activities,
    on="activity_id",
    how="left",
    suffixes=("", "_activity")
)


print("Merged Dataset:", data.shape)


# ============================================================
# 8. CREATE TARGET
# ============================================================

interaction_weights = {
    "rating": 1.0,
    "visit": 0.80,
    "save": 0.65,
    "view": 0.25
}


data["interaction_weight"] = (
    data["interaction_type"]
    .map(interaction_weights)
    .fillna(0.1)
)


data["rating_score"] = (
    data["rating"] / 5
)


data["target_score"] = (
    data["interaction_weight"] * 0.45 +
    data["rating_score"] * 0.55
)


# Positive interaction

data["target"] = (
    (
        (data["rating"] >= 4) |
        (data["interaction_type"].isin(["visit", "save"]))
    )
    .astype(int)
)


print("\nTarget Distribution")
print(data["target"].value_counts())


# ============================================================
# 9. TEMPORAL FEATURES
# ============================================================

data["year"] = data["event_date"].dt.year
data["month"] = data["event_date"].dt.month
data["day_of_week"] = data["event_date"].dt.dayofweek

data["is_weekend"] = (
    data["day_of_week"] >= 5
).astype(int)


# ============================================================
# 10. DIRECT USER-ACTIVITY FEATURES
# ============================================================

data["category_match"] = (
    data["preferred_categories"]
    .fillna("")
    .astype(str)
    .str.lower()
    .apply(
        lambda x: 1
        if any(
            c.strip() != "" and
            c.strip() in x.split("|")
            for c in [""]
        )
        else 0
    )
)


# Correct category matching

def category_match(row):

    prefs = str(
        row.get("preferred_categories", "")
    ).lower()

    category = str(
        row.get("category", "")
    ).lower()

    if not prefs or not category:
        return 0

    return int(
        category in [
            x.strip()
            for x in prefs.split("|")
        ]
    )


data["category_match"] = data.apply(
    category_match,
    axis=1
)


# ============================================================
# 11. BUDGET MATCH
# ============================================================

data["budget_match"] = (
    data["avg_cost"] <=
    pd.to_numeric(
        data["budget_usd"],
        errors="coerce"
    ).fillna(0)
).astype(int)


# ============================================================
# 12. INDOOR / OUTDOOR MATCH
# ============================================================

def indoor_match(row):

    preference = str(
        row.get(
            "indoor_outdoor_preference",
            ""
        )
    ).lower()

    activity_type = str(
        row.get(
            "indoor_outdoor",
            ""
        )
    ).lower()

    if preference == "mixed":
        return 1

    if preference == activity_type:
        return 1

    return 0


data["indoor_match"] = data.apply(
    indoor_match,
    axis=1
)


# ============================================================
# 13. GROUP MATCH
# ============================================================

def group_match(row):

    user_group = str(
        row.get("group_type", "")
    ).lower()

    activity_group = str(
        row.get("group_suitability", "")
    ).lower()

    if not user_group or not activity_group:
        return 0

    if user_group in activity_group:
        return 1

    if activity_group in ["all", "mixed"]:
        return 1

    return 0


data["group_match"] = data.apply(
    group_match,
    axis=1
)


# ============================================================
# 14. KIDS MATCH
# ============================================================

data["kids_match"] = (
    (
        data["has_kids_bool"] &
        data["kids_suitable_bool"]
    ) |
    (~data["has_kids_bool"])
).astype(int)


# ============================================================
# 15. WHEELCHAIR MATCH
# ============================================================

data["wheelchair_match"] = (
    (
        ~data["wheelchair_need_bool"]
    ) |
    (
        data["wheelchair_need_bool"] &
        data["wheelchair_bool"]
    )
).astype(int)


# ============================================================
# 16. CITY MATCH
# ============================================================

data["city_match"] = (
    data["city"] ==
    data["home_city"]
).astype(int)


# ============================================================
# 17. WEATHER FEATURES
# ============================================================

weather_small = weather[
    [
        "weather_condition",
        "preferred_indoor_outdoor",
        "recommendation_adjustment"
    ]
].copy()


weather_small["weather_condition"] = (
    weather_small["weather_condition"]
    .str.lower()
    .str.strip()
)


data = data.merge(
    weather_small,
    left_on="weather_context",
    right_on="weather_condition",
    how="left"
)


data["recommendation_adjustment"] = (
    pd.to_numeric(
        data["recommendation_adjustment"],
        errors="coerce"
    ).fillna(0)
)


data["weather_match"] = (
    (
        data["preferred_indoor_outdoor"]
        .fillna("")
        .str.lower()
        ==
        data["indoor_outdoor"]
        .fillna("")
        .str.lower()
    )
).astype(int)


# ============================================================
# 18. SORT BY DATE
# ============================================================

data = data.sort_values(
    "event_date"
).reset_index(drop=True)


# ============================================================
# 19. TEMPORAL TRAIN / TEST SPLIT
# ============================================================

split_index = int(
    len(data) * 0.82
)


train_data = data.iloc[
    :split_index
].copy()


test_data = data.iloc[
    split_index:
].copy()


print("\nTemporal Split")
print("Train:", train_data.shape)
print("Test :", test_data.shape)


# ============================================================
# 20. USER STATISTICS
# ============================================================

user_stats = (
    train_data
    .groupby("user_id")
    .agg(
        user_interaction_count=(
            "activity_id",
            "count"
        ),
        user_positive_count=(
            "target",
            "sum"
        ),
        user_avg_rating=(
            "rating",
            "mean"
        ),
        user_rating_count=(
            "rating",
            lambda x: (x > 0).sum()
        )
    )
    .reset_index()
)


user_stats["user_positive_rate"] = (
    user_stats["user_positive_count"] /
    user_stats["user_interaction_count"].clip(lower=1)
)


# ============================================================
# 21. USER CATEGORY PROFILE
# ============================================================

user_category = (
    train_data
    .groupby(
        [
            "user_id",
            "category"
        ]
    )
    .agg(
        user_category_interactions=(
            "activity_id",
            "count"
        ),
        user_category_positive=(
            "target",
            "sum"
        ),
        user_category_rating=(
            "rating",
            "mean"
        )
    )
    .reset_index()
)


user_category[
    "user_category_positive_rate"
] = (
    user_category[
        "user_category_positive"
    ] /
    user_category[
        "user_category_interactions"
    ].clip(lower=1)
)


# ============================================================
# 22. USER INDOOR / OUTDOOR PROFILE
# ============================================================

user_indoor = (
    train_data
    .groupby(
        [
            "user_id",
            "indoor_outdoor"
        ]
    )
    .agg(
        user_indoor_interactions=(
            "activity_id",
            "count"
        ),
        user_indoor_positive=(
            "target",
            "mean"
        )
    )
    .reset_index()
)


# ============================================================
# 23. ACTIVITY PROFILE
# ============================================================

activity_stats = (
    train_data
    .groupby("activity_id")
    .agg(
        activity_interaction_count=(
            "user_id",
            "count"
        ),
        activity_positive_count=(
            "target",
            "sum"
        ),
        activity_avg_rating=(
            "rating",
            "mean"
        )
    )
    .reset_index()
)


activity_stats[
    "activity_positive_rate"
] = (
    activity_stats[
        "activity_positive_count"
    ] /
    activity_stats[
        "activity_interaction_count"
    ].clip(lower=1)
)


# ============================================================
# 24. SMOOTH ACTIVITY SCORE
# ============================================================

global_positive_rate = (
    train_data["target"].mean()
)


m = 10


activity_stats[
    "activity_smoothed_score"
] = (
    (
        activity_stats["activity_positive_count"]
        +
        m * global_positive_rate
    )
    /
    (
        activity_stats[
            "activity_interaction_count"
        ]
        + m
    )
)


# ============================================================
# 25. CATEGORY STATISTICS
# ============================================================

category_stats = (
    train_data
    .groupby("category")
    .agg(
        category_interaction_count=(
            "activity_id",
            "count"
        ),
        category_positive_rate=(
            "target",
            "mean"
        ),
        category_avg_rating=(
            "rating",
            "mean"
        )
    )
    .reset_index()
)


category_stats[
    "category_smoothed_score"
] = (
    (
        category_stats[
            "category_positive_rate"
        ] *
        category_stats[
            "category_interaction_count"
        ]
        +
        global_positive_rate * m
    )
    /
    (
        category_stats[
            "category_interaction_count"
        ] + m
    )
)


# ============================================================
# 26. FEATURE CREATION FUNCTION
# ============================================================

def build_features(df):

    result = df.copy()

    # user profile

    result = result.merge(
        user_stats,
        on="user_id",
        how="left"
    )


    # user category profile

    result = result.merge(
        user_category[
            [
                "user_id",
                "category",
                "user_category_interactions",
                "user_category_positive_rate",
                "user_category_rating"
            ]
        ],
        on=[
            "user_id",
            "category"
        ],
        how="left"
    )


    # user indoor profile

    result = result.merge(
        user_indoor,
        on=[
            "user_id",
            "indoor_outdoor"
        ],
        how="left"
    )


    # activity profile

    result = result.merge(
        activity_stats[
            [
                "activity_id",
                "activity_interaction_count",
                "activity_positive_rate",
                "activity_avg_rating",
                "activity_smoothed_score"
            ]
        ],
        on="activity_id",
        how="left"
    )


    # category profile

    result = result.merge(
        category_stats,
        on="category",
        how="left"
    )


    # fill numeric missing values

    numeric_cols = [
        "user_interaction_count",
        "user_positive_count",
        "user_avg_rating",
        "user_rating_count",
        "user_positive_rate",
        "user_category_interactions",
        "user_category_positive_rate",
        "user_category_rating",
        "user_indoor_interactions",
        "user_indoor_positive",
        "activity_interaction_count",
        "activity_positive_rate",
        "activity_avg_rating",
        "activity_smoothed_score",
        "category_interaction_count",
        "category_positive_rate",
        "category_avg_rating",
        "category_smoothed_score"
    ]


    for col in numeric_cols:

        if col not in result.columns:
            result[col] = 0

        result[col] = pd.to_numeric(
            result[col],
            errors="coerce"
        ).fillna(0)


    return result


print("\nCreating feature datasets...")


train_features = build_features(
    train_data
)


test_features = build_features(
    test_data
)


print(
    "Train feature dataset:",
    train_features.shape
)


print(
    "Test feature dataset:",
    test_features.shape
)


# ============================================================
# 27. ADVANCED FEATURES
# ============================================================

for df in [
    train_features,
    test_features
]:

    df["rating_strength"] = (
        df["rating"] / 5
    )


    df["behavior_score"] = (
        df["interaction_weight"]
        * 0.5
        +
        df["rating_strength"]
        * 0.5
    )


    df["personalization_score"] = (
        df["category_match"] * 0.30
        +
        df["budget_match"] * 0.20
        +
        df["indoor_match"] * 0.10
        +
        df["group_match"] * 0.10
        +
        df["kids_match"] * 0.10
        +
        df["wheelchair_match"] * 0.10
        +
        df["city_match"] * 0.10
    )


    df["activity_quality"] = (
        df["activity_smoothed_score"] * 0.7
        +
        df["data_quality_score"].fillna(0) * 0.3
    )


# ============================================================
# 28. FEATURES
# ============================================================

feature_columns = [
    "category_match",
    "budget_match",
    "indoor_match",
    "group_match",
    "kids_match",
    "wheelchair_match",
    "city_match",
    "weather_match",

    "avg_cost",
    "typical_duration_min",

    "recommendation_adjustment",
    "data_quality_score",

    "month",
    "day_of_week",
    "is_weekend",

    "interaction_weight",
    "rating_strength",
    "behavior_score",

    "personalization_score",
    "activity_quality",

    "user_interaction_count",
    "user_positive_count",
    "user_avg_rating",
    "user_rating_count",
    "user_positive_rate",

    "user_category_interactions",
    "user_category_positive_rate",
    "user_category_rating",

    "user_indoor_interactions",
    "user_indoor_positive",

    "activity_interaction_count",
    "activity_positive_rate",
    "activity_avg_rating",
    "activity_smoothed_score",

    "category_interaction_count",
    "category_positive_rate",
    "category_avg_rating",
    "category_smoothed_score"
]


feature_columns = [
    col
    for col in feature_columns
    if col in train_features.columns
]


X_train = train_features[
    feature_columns
].copy()


y_train = train_features[
    "target"
].copy()


X_test = test_features[
    feature_columns
].copy()


y_test = test_features[
    "target"
].copy()


# ============================================================
# 29. HANDLE CLASS IMBALANCE
# ============================================================

positive = y_train.sum()
negative = len(y_train) - positive


scale_pos_weight = (
    negative / positive
)


print(
    "\nPositive:",
    positive
)

print(
    "Negative:",
    negative
)

print(
    "Scale Pos Weight:",
    round(
        scale_pos_weight,
        4
    )
)


# ============================================================
# 30. XGBOOST MODEL
# ============================================================

model = XGBClassifier(

    n_estimators=700,

    max_depth=6,

    learning_rate=0.035,

    min_child_weight=5,

    subsample=0.85,

    colsample_bytree=0.85,

    gamma=0.15,

    reg_alpha=0.15,

    reg_lambda=2.0,

    objective="binary:logistic",

    eval_metric="logloss",

    scale_pos_weight=scale_pos_weight,

    random_state=42,

    n_jobs=-1
)


print("\nTraining XGBoost...")


model.fit(
    X_train,
    y_train,

    eval_set=[
        (
            X_test,
            y_test
        )
    ],

    verbose=False
)


# ============================================================
# 31. MODEL EVALUATION
# ============================================================

probabilities = model.predict_proba(
    X_test
)[:, 1]


predictions = (
    probabilities >= 0.5
).astype(int)


accuracy = accuracy_score(
    y_test,
    predictions
)


precision = precision_score(
    y_test,
    predictions,
    zero_division=0
)


recall = recall_score(
    y_test,
    predictions,
    zero_division=0
)


f1 = f1_score(
    y_test,
    predictions,
    zero_division=0
)


roc_auc = roc_auc_score(
    y_test,
    probabilities
)


pr_auc = average_precision_score(
    y_test,
    probabilities
)


print("\n")
print("=" * 70)
print("MODEL PERFORMANCE")
print("=" * 70)

print(
    f"Accuracy : {accuracy:.4f}"
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"F1 Score : {f1:.4f}"
)

print(
    f"ROC-AUC  : {roc_auc:.4f}"
)

print(
    f"PR-AUC   : {pr_auc:.4f}"
)


print("\nClassification Report")
print(
    classification_report(
        y_test,
        predictions,
        zero_division=0
    )
)


# ============================================================
# 32. FEATURE IMPORTANCE
# ============================================================

importance = pd.DataFrame({

    "feature":
        feature_columns,

    "importance":
        model.feature_importances_

})


importance = importance.sort_values(
    "importance",
    ascending=False
)


print("\nTop Features")

print(
    importance.head(25).to_string(
        index=False
    )
)


# ============================================================
# 33. RECOMMENDATION FUNCTION
# ============================================================

def recommend(
    user_id,
    weather_condition="sunny",
    top_n=10
):

    user_rows = users[
        users["user_id"] == user_id
    ]


    if len(user_rows) == 0:

        print(
            "User not found."
        )

        return pd.DataFrame()


    user = user_rows.iloc[0]


    candidates = activities.copy()


    # --------------------------------------------------------
    # Remove already interacted activities
    # --------------------------------------------------------

    seen = set(
        interactions.loc[
            interactions["user_id"] == user_id,
            "activity_id"
        ].astype(str)
    )


    candidates = candidates[
        ~candidates[
            "activity_id"
        ].astype(str).isin(seen)
    ].copy()


    if len(candidates) == 0:

        return pd.DataFrame()


    # --------------------------------------------------------
    # User preference
    # --------------------------------------------------------

    preferences = str(
        user.get(
            "preferred_categories",
            ""
        )
    ).lower()


    preferred_categories = [
        x.strip()
        for x in preferences.split("|")
        if x.strip()
    ]


    candidates["category_match"] = (
        candidates["category"]
        .isin(preferred_categories)
        .astype(int)
    )


    # --------------------------------------------------------
    # Budget
    # --------------------------------------------------------

    budget = pd.to_numeric(
        user.get(
            "budget_usd",
            0
        ),
        errors="coerce"
    )


    if pd.isna(budget):
        budget = 0


    candidates["budget_match"] = (
        candidates["avg_cost"] <= budget
    ).astype(int)


    # --------------------------------------------------------
    # Indoor / Outdoor
    # --------------------------------------------------------

    preference = str(
        user.get(
            "indoor_outdoor_preference",
            "mixed"
        )
    ).lower()


    if preference == "mixed":

        candidates["indoor_match"] = 1

    else:

        candidates["indoor_match"] = (
            candidates["indoor_outdoor"] ==
            preference
        ).astype(int)


    # --------------------------------------------------------
    # Group
    # --------------------------------------------------------

    group = str(
        user.get(
            "group_type",
            ""
        )
    ).lower()


    candidates["group_match"] = (
        candidates[
            "group_suitability"
        ].str.contains(
            group,
            case=False,
            na=False
        )
    ).astype(int)


    # --------------------------------------------------------
    # Kids
    # --------------------------------------------------------

    has_kids = to_bool(
        user.get(
            "has_kids_bool",
            False
        )
    )


    if has_kids:

        candidates["kids_match"] = (
            candidates[
                "kids_suitable_bool"
            ]
        ).astype(int)

    else:

        candidates["kids_match"] = 1


    # --------------------------------------------------------
    # Wheelchair
    # --------------------------------------------------------

    wheelchair_need = to_bool(
        user.get(
            "wheelchair_need_bool",
            False
        )
    )


    if wheelchair_need:

        candidates["wheelchair_match"] = (
            candidates[
                "wheelchair_bool"
            ]
        ).astype(int)

    else:

        candidates["wheelchair_match"] = 1


    # --------------------------------------------------------
    # City
    # --------------------------------------------------------

    home_city = str(
        user.get(
            "home_city",
            ""
        )
    ).lower()


    candidates["city_match"] = (
        candidates["city"].str.lower()
        ==
        home_city
    ).astype(int)


    # --------------------------------------------------------
    # Weather
    # --------------------------------------------------------

    weather_row = weather[
        weather[
            "weather_condition"
        ].str.lower()
        ==
        str(
            weather_condition
        ).lower()
    ]


    weather_adjustment = 0


    if len(weather_row) > 0:

        weather_row = weather_row.iloc[0]

        preferred_weather_type = str(
            weather_row[
                "preferred_indoor_outdoor"
            ]
        ).lower()


        weather_adjustment = float(
            weather_row[
                "recommendation_adjustment"
            ]
        )


        candidates["weather_match"] = (
            candidates[
                "indoor_outdoor"
            ].str.lower()
            ==
            preferred_weather_type
        ).astype(int)

    else:

        candidates["weather_match"] = 0


    candidates[
        "recommendation_adjustment"
    ] = weather_adjustment


    # --------------------------------------------------------
    # Basic defaults
    # --------------------------------------------------------

    candidates["month"] = 8
    candidates["day_of_week"] = 5
    candidates["is_weekend"] = 1


    candidates["interaction_weight"] = 0
    candidates["rating_strength"] = 0


    # --------------------------------------------------------
    # Merge activity statistics
    # --------------------------------------------------------

    candidates = candidates.merge(
        activity_stats[
            [
                "activity_id",
                "activity_interaction_count",
                "activity_positive_rate",
                "activity_avg_rating",
                "activity_smoothed_score"
            ]
        ],
        on="activity_id",
        how="left"
    )


    candidates = candidates.merge(
        category_stats,
        on="category",
        how="left"
    )


    # --------------------------------------------------------
    # User statistics
    # --------------------------------------------------------

    ustat = user_stats[
        user_stats[
            "user_id"
        ] == user_id
    ]


    if len(ustat) > 0:

        ustat = ustat.iloc[0]

        for col in [
            "user_interaction_count",
            "user_positive_count",
            "user_avg_rating",
            "user_rating_count",
            "user_positive_rate"
        ]:

            candidates[col] = ustat[
                col
            ]

    else:

        for col in [
            "user_interaction_count",
            "user_positive_count",
            "user_avg_rating",
            "user_rating_count",
            "user_positive_rate"
        ]:

            candidates[col] = 0


    # --------------------------------------------------------
    # User category profile
    # --------------------------------------------------------

    uc = user_category[
        user_category["user_id"] == user_id
    ][
        [
            "category",
            "user_category_interactions",
            "user_category_positive_rate",
            "user_category_rating"
        ]
    ]


    candidates = candidates.merge(
        uc,
        on="category",
        how="left"
    )


    for col in [
        "user_category_interactions",
        "user_category_positive_rate",
        "user_category_rating"
    ]:

        candidates[col] = (
            pd.to_numeric(
                candidates[col],
                errors="coerce"
            )
            .fillna(0)
        )


    # --------------------------------------------------------
    # User indoor profile
    # --------------------------------------------------------

    ui = user_indoor[
        user_indoor["user_id"] == user_id
    ][
        [
            "indoor_outdoor",
            "user_indoor_interactions",
            "user_indoor_positive"
        ]
    ]


    candidates = candidates.merge(
        ui,
        on="indoor_outdoor",
        how="left"
    )


    for col in [
        "user_indoor_interactions",
        "user_indoor_positive"
    ]:

        candidates[col] = (
            pd.to_numeric(
                candidates[col],
                errors="coerce"
            )
            .fillna(0)
        )


    # --------------------------------------------------------
    # Fill missing values
    # --------------------------------------------------------

    for col in feature_columns:

        if col not in candidates.columns:

            candidates[col] = 0


        candidates[col] = pd.to_numeric(
            candidates[col],
            errors="coerce"
        ).fillna(0)


    # --------------------------------------------------------
    # Advanced features
    # --------------------------------------------------------

    candidates["behavior_score"] = 0


    candidates["personalization_score"] = (
        candidates["category_match"] * 0.30
        +
        candidates["budget_match"] * 0.20
        +
        candidates["indoor_match"] * 0.10
        +
        candidates["group_match"] * 0.10
        +
        candidates["kids_match"] * 0.10
        +
        candidates["wheelchair_match"] * 0.10
        +
        candidates["city_match"] * 0.10
    )


    candidates["activity_quality"] = (
        candidates[
            "activity_smoothed_score"
        ] * 0.7
        +
        candidates[
            "data_quality_score"
        ] * 0.3
    )


    # --------------------------------------------------------
    # Model prediction
    # --------------------------------------------------------

    probabilities = model.predict_proba(
        candidates[
            feature_columns
        ]
    )[:, 1]


    candidates["ai_probability"] = (
        probabilities
    )


    # --------------------------------------------------------
    # Practical recommendation score
    # --------------------------------------------------------

    candidates["final_score"] = (

        candidates[
            "ai_probability"
        ] * 0.55

        +

        candidates[
            "category_match"
        ] * 0.12

        +

        candidates[
            "budget_match"
        ] * 0.10

        +

        candidates[
            "city_match"
        ] * 0.07

        +

        candidates[
            "indoor_match"
        ] * 0.05

        +

        candidates[
            "group_match"
        ] * 0.04

        +

        candidates[
            "weather_match"
        ] * 0.04

        +

        candidates[
            "activity_quality"
        ] * 0.03
    )


    # Strong constraints

    suitable = candidates[
        candidates["budget_match"] == 1
    ].copy()


    if wheelchair_need:

        suitable = suitable[
            suitable[
                "wheelchair_match"
            ] == 1
        ]


    if has_kids:

        suitable = suitable[
            suitable[
                "kids_match"
            ] == 1
        ]


    if len(suitable) < top_n:

        suitable = candidates.copy()


    # --------------------------------------------------------
    # Diversity ranking
    # --------------------------------------------------------

    suitable = suitable.sort_values(
        "final_score",
        ascending=False
    )


    selected = []

    category_count = {}

    max_same_category = max(
        2,
        int(
            np.ceil(
                top_n * 0.35
            )
        )
    )


    for _, row in suitable.iterrows():

        category = row["category"]

        count = category_count.get(
            category,
            0
        )


        if count >= max_same_category:

            continue


        selected.append(row)

        category_count[
            category
        ] = count + 1


        if len(selected) >= top_n:

            break


    result = pd.DataFrame(
        selected
    )


    if len(result) == 0:

        return result


    return result[
        [
            "activity_id",
            "name",
            "category",
            "subcategory",
            "city",
            "state",
            "indoor_outdoor",
            "avg_cost",
            "typical_duration_min",
            "ai_probability",
            "final_score"
        ]
    ].reset_index(
        drop=True
    )


# ============================================================
# 34. DISPLAY RECOMMENDATIONS
# ============================================================

def show_recommendations(
    user_id,
    weather_condition,
    top_n=10
):

    print("\n")
    print("=" * 70)

    print(
        f"RECOMMENDATIONS FOR {user_id}"
    )

    print(
        f"Weather: {weather_condition}"
    )

    print("=" * 70)


    result = recommend(
        user_id,
        weather_condition,
        top_n
    )


    if len(result) == 0:

        print(
            "No suitable recommendations found."
        )

        return


    for i, row in result.iterrows():

        print(
            f"\n{i + 1}. {row['name']}"
        )

        print(
            f"   Category: {row['category']}"
        )

        print(
            f"   Subcategory: {row['subcategory']}"
        )

        print(
            f"   Location: "
            f"{row['city']}, "
            f"{row['state']}"
        )

        print(
            f"   Type: "
            f"{row['indoor_outdoor']}"
        )

        print(
            f"   Estimated Cost: "
            f"${row['avg_cost']:.2f}"
        )

        print(
            f"   Duration: "
            f"{row['typical_duration_min']:.0f} min"
        )

        print(
            f"   AI Probability: "
            f"{row['ai_probability']:.3f}"
        )

        print(
            f"   Final Score: "
            f"{row['final_score']:.3f}"
        )


# ============================================================
# 35. TEST RECOMMENDATION
# ============================================================

show_recommendations(
    "syn_u0001",
    "sunny",
    10
)


show_recommendations(
    "syn_u0001",
    "rain",
    10
)


print("\n")
print("=" * 70)
print("MODEL TRAINING COMPLETED")
print("=" * 70)

WEEKEND ACTIVITY PLANNER - AI RECOMMENDER

Dataset Shapes
Users: (1000, 10)
Activities: (11630, 23)
Interactions: (15695, 9)
Weather: (5, 4)

Preparing recommendation dataset...
Merged Dataset: (15695, 45)

Target Distribution
target
1    14382
0     1313
Name: count, dtype: int64

Temporal Split
Train: (12869, 64)
Test : (2826, 64)

Creating feature datasets...
Train feature dataset: (12869, 82)
Test feature dataset: (2826, 82)

Positive: 11806
Negative: 1063
Scale Pos Weight: 0.09

Training XGBoost...


MODEL PERFORMANCE
Accuracy : 0.9161
Precision: 0.9323
Recall   : 0.9790
F1 Score : 0.9551
ROC-AUC  : 0.9050
PR-AUC   : 0.9900

Classification Report
              precision    recall  f1-score   support

           0       0.55      0.27      0.36       250
           1       0.93      0.98      0.96      2576

    accuracy                           0.92      2826
   macro avg       0.74      0.62      0.66      2826
weighted avg       0.90      0.92      0.90      2826


Top Features

In [5]:
joblib.dump(model,"weekend_model.pkl")

['weekend_model.pkl']